In [1]:
import sys
sys.path.insert(0,'../g3algo/')
from g3groupfinder import giantmodel, decayexp, sigmarange
import foftools as fof
import iterativecombination as ic
from smoothedbootstrap import smoothedbootstrap as sbs
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import MaxNLocator
from matplotlib import rcParams
from scipy.optimize import curve_fit
from center_binned_stats import center_binned_stats
from scipy.stats import ks_2samp as kstest
from scipy.interpolate import interp1d
rcParams['axes.labelsize'] = 9
rcParams['xtick.labelsize'] = 9
rcParams['ytick.labelsize'] = 9
rcParams['legend.fontsize'] = 9
rcParams['font.family'] = 'sans-serif'
rcParams['grid.color'] = 'k'
rcParams['grid.linewidth'] = 0.2
my_locator = MaxNLocator(6)
singlecolsize = (3.3522420091324205, 2.0717995001590714)
doublecolsize = (7.100005949910059, 4.3880449973709)
HUBBLE_CONST = 70.
import survey_volume as sv

from scipy.spatial.distance import euclidean
def hellinger2(p, q):
    return euclidean(np.sqrt(p), np.sqrt(q)) / np.sqrt(2)

In [3]:
ecodata = pd.read_csv("ECOdata_G3catalog_luminosity.csv")
ecodata = ecodata[(ecodata.dup<1)&(ecodata.cz<7470)&(ecodata.cz>2530)]
print(len(ecodata[ecodata.absrmag<-17.33]))
print(len(ecodata[ecodata.absrmag<-19.5]))
#print(len(ecodata[ecodata.absrmag>=-19.5]))
print(len(ecodata[(ecodata.absrmag<-17.33)&(ecodata.g3grpcz_l>3000)&(ecodata.g3grpcz_l<7000)]))

12649
4625
9550


In [4]:
sv.comoving_volume(130.05,237.45,-1,49.85,2530/3e+5,7470/3e5,100,0.3,0.7)

191936.0761239577

In [8]:
resdata = pd.read_csv("RESOLVEdata_G3catalog_luminosity.csv")
#resdata = resdata[(resdata.cz<7250)&(resdata.cz>4250)]
inlum = ((resdata.f_b>0)&(resdata.absrmag<-17.)|((resdata.f_a>0)&(resdata.absrmag<-17.33)))
print(len(resdata[inlum]))
print(len(resdata[resdata.absrmag<-19.5]))
print(len(resdata[(inlum)&(resdata.g3grpcz_l>4500)&(resdata.g3grpcz_l<7000)]))

1670
564
1449


In [10]:
min(resdata.cz),max(resdata.cz)

(4253.5, 7283.9)

In [5]:
resdata = pd.read_csv("RESOLVEdata_G3catalog_luminosity.csv")
len(resdata[resdata.fl_insample>0])

1443

In [6]:
sv.comoving_volume(131.25,236.25,0,5,4250/3e+5,7250/3e5,100,0.3,0.7) # RESOLVE-A

15876.972018379183

In [7]:
sv.comoving_volume(-30,45,-1.25,1.25,4250/3e+5,7250/3e5,100,0.3,0.7) # RESOLVE-b

5677.100201133777

In [8]:
resdata = pd.read_csv("RESOLVEdata_080822.csv")
len(resdata[(resdata.fl_insample>0)])

1443

In [9]:
inlum = ((resdata.f_b>0) & (resdata.absrmag<-17)) | ((resdata.f_a>0) & (resdata.absrmag < -17.33)) 
len(resdata[inlum])

1670

In [10]:
insample = inlum & (resdata.grpcz>4500) & (resdata.grpcz<7000) 
len(resdata[insample])

1443

what galaxy is missing?

In [38]:
from scipy.io import readsav

In [39]:
cielo = readsav('/srv/one/resolve/database_internal/merged_idl_catalog/stable/resolvecatalog.dat')
cielophot = readsav('/srv/one/resolve/database_internal/merged_idl_catalog/stable/resolvecatalogphot.dat')

In [40]:
cielo_absrmag = np.float64(cielophot['absmagr'])
cielo_fa = np.float64(cielo['inspring'])
cielo_fb = np.float64(cielo['infall'])
cielo_grpcz = np.float64(cielo['groupcz'])
cielo_name = np.array([x.decode('utf-8') for x in cielo['name']])

In [42]:
cielo_inlum = ((cielo_fb>0) & (cielo_absrmag<=-17)) | ((cielo_fa>0) & (cielo_absrmag<=-17.33))

In [43]:
cielo_names_1671 = cielo_name[cielo_inlum]

In [44]:
resdata = pd.read_csv("RESOLVEdata_080822.csv")
inlum = ((resdata.f_b>0) & (resdata.absrmag<=-17)) | ((resdata.f_a>0) & (resdata.absrmag <= -17.33)) 
resdata = resdata[inlum]

In [45]:
wiki_names_1670 = resdata.name.to_numpy()

In [46]:
wiki_names_1670

array(['rf0001', 'rf0002', 'rf0003', ..., 'rs1462', 'rs1463', 'rs1518'],
      dtype=object)

In [47]:
for nm in cielo_names_1671:
    if nm not in wiki_names_1670:
        print(nm)

In [48]:
print(len(cielo_names_1671),len(wiki_names_1670))

1671 1674


In [36]:
resdata = pd.read_csv("RESOLVEdata_080822.csv")
resdata[resdata.name=='rf0725'].absrmag

718   -17.0
Name: absrmag, dtype: float64

In [37]:
cielo_absrmag[cielo_name=='rf0725']

array([-17.00415039])

In [49]:
resdata.absrmag

0      -17.68
1      -18.62
2      -17.65
3      -20.29
4      -21.18
        ...  
2227   -17.51
2228   -17.48
2229   -18.89
2230   -17.50
2285   -17.37
Name: absrmag, Length: 1674, dtype: float64